# Email Sending

## 1 - Raw SMTP (standard library)

### Using Google SMTP

- For this setup, we need to generate the `Google App Password`.

- The Gmail Account must be 2-FA (Two Factor Authication) protected to generate the Google App Password.

- Rename the `.env.sample` file to `.env` and enter the values for each environment variable.

- `SMTP_HOST` is google's SMTP host name which is `smtp.gmail.com`.

- `SMTP_PORT` is google's SMTP port which is `587`.

- `SMTP_USER` is the email address used to generate the `Google App Password`.

- `SMTP_PASS` is the password provided by the `Google App Password` manager.

In [ ]:
# ! pip install email-validator # uncomment this to install package via pip
# ! pip install python-dotenv # uncomment this to install package via pip
# ! pip install pandas # uncomment this to install package via pip
# ! pip install yagmail # uncomment this to install package via pip
# ! pip install aiosmtplib # uncomment this to install package via pip
! uv add email-validator
! uv add python-dotenv
! uv add pandas
! uv add yagmail
! uv add aiosmtplib

In [ ]:
# imports
import os
import smtplib
import ssl
import time
from email.message import EmailMessage
from typing import Any, List, Optional

import aiosmtplib
import yagmail
from dotenv import load_dotenv
from yagmail.sender import Client

In [ ]:
# constants

SMTP_HOST: str = os.getenv("SMTP_HOST", "")
SMTP_PORT: int = int(os.getenv("SMTP_PORT", 0))
SMTP_USER: str = os.getenv("SMTP_USER", "")
SMTP_PASS: str = os.getenv("SMTP_PASS", "")

RETRY_ATTEMPTS: int = 3
RETRY_BACKOFF: int = 2  # in seconds

In [ ]:
# load environment variables
load_dotenv(".env")

**1 - SMTPEmail Sender Using RAW SMTP Library**

In [ ]:
class SMTPEmailSender:
    """Sends emails via SMTP with retry logic and TLS security.

    Provides a production-grade SMTP client with automatic retry,
    exponential backoff, and TLS encryption.
    """

    def __init__(self, host: str, port: int, user: str, password: str):
        """Initializes the sender with SMTP server credentials.

        Args:
            host: SMTP server hostname.
            port: SMTP server port.
            user: Email account username.
            password: Email account password or app password.
        """
        self.host = host
        self.port = port
        self.user = user
        self.password = password

    def _build_message(
        self, subject: str, body: str, to: List[str], cc: Optional[List[str]] = None
    ) -> EmailMessage:
        """Builds an EmailMessage with the given parameters.

        Args:
            subject: Email subject line.
            body: Plain-text email body.
            to: List of primary recipient addresses.
            cc: Optional list of CC recipient addresses.

        Returns:
            A fully constructed EmailMessage instance.
        """
        email_message: EmailMessage = EmailMessage()
        email_message["From"] = self.user
        email_message["To"] = ", ".join(to)
        if cc:
            email_message["Cc"] = ", ".join(cc)
        email_message["Subject"] = subject
        email_message.set_content(body)
        return email_message

    def send_email(
        self, subject: str, body: str, to: List[str], cc: Optional[List[str]] = None
    ) -> None:
        """Sends an email with retry and backoff via a secure TLS connection.

        Establishes a TLS-secured connection to the SMTP server, authenticates,
        and sends the message. Retries on failure with exponential backoff.

        Args:
            subject: Email subject line.
            body: Plain-text email body.
            to: List of primary recipient addresses.
            cc: Optional list of CC recipient addresses.

        Raises:
            RuntimeError: If all retry attempts fail.
        """
        email_message: EmailMessage = self._build_message(
            subject=subject, body=body, to=to, cc=cc
        )
        recipients: List[str] = to + (cc or [])

        # create secure channel with the smtp
        context: ssl.SSLContext = ssl.create_default_context()

        previous_exception = None

        for attempt in range(1, RETRY_ATTEMPTS + 1):
            try:
                with smtplib.SMTP(host=self.host, port=self.port, timeout=10) as server:
                    server.ehlo()  # this is optional if we don't add this, the new features of the smtp will not be available
                    server.starttls(context=context)
                    server.ehlo()
                    server.login(self.user, self.password)
                    server.send_message(
                        msg=email_message, from_addr=self.user, to_addrs=recipients
                    )
                return  # success
            except Exception as e:
                previous_exception = e
                time.sleep(RETRY_BACKOFF * attempt)
        raise RuntimeError(f"Email sending failed after retries: {previous_exception}")

**2 - Email Sending Using `yagmail`**

In [ ]:
class YaGmailSender:
    """Sends emails using the yagmail library with retry logic.

    Wraps the yagmail SMTP client to add automatic retry
    and exponential backoff on failure.
    """

    def __init__(self, user: str, password: str) -> None:
        """Initializes the yagmail client with the given credentials.

        Args:
            user: Gmail email address.
            password: Google App Password.
        """
        self.user = user
        self.password = password
        self.client: Client = yagmail.SMTP(user=user, password=password)

    def send_email(
        self, subject: str, body: Any, to: List[str], cc: Optional[List[str]] = None
    ) -> None:
        """Sends an email with retry and backoff via yagmail.

        Delegates to the yagmail client for sending. Retries on failure
        with exponential backoff.

        Args:
            subject: Email subject line.
            body: Email body content (plain text or HTML).
            to: List of recipient addresses.
            cc: Optional list of CC recipient addresses.

        Raises:
            RuntimeError: If all retry attempts fail.
        """
        previous_exception = None

        for attempt in range(1, RETRY_ATTEMPTS + 1):
            try:
                self.client.send(to=to, subject=[subject], cc=cc, contents=body)
                return
            except Exception as e:
                previous_exception = e
                time.sleep(attempt * 2)
        raise RuntimeError(f"YaGmail send failed: {previous_exception}")

**3 - Async Email Sending (aiosmtplib)**

_When to use:_
- `FastAPI` / `asyncio services`

- High concurrency email dispatching

In [ ]:
class AsyncEmailSender:
    """Sends emails asynchronously using aiosmtplib.

    Provides an async SMTP client for use in asyncio-based
    applications (e.g. FastAPI, high-concurrency services).
    """

    def __init__(self, host: str, port: int, user: str, password: str) -> None:
        """Initializes the async sender with SMTP server credentials.

        Args:
            host: SMTP server hostname.
            port: SMTP server port.
            user: Email account username.
            password: Email account password or app password.
        """
        self.host = host
        self.port = port
        self.user = user
        self.password = password

    def _build_message(self, subject: str, body: str, to: list[str]) -> EmailMessage:
        """Builds an EmailMessage with the given parameters.

        Args:
            subject: Email subject line.
            body: Plain-text email body.
            to: List of primary recipient addresses.

        Returns:
            A fully constructed EmailMessage instance.
        """
        email_message: EmailMessage = EmailMessage()
        email_message["From"] = self.user
        email_message["To"] = ", ".join(to)
        email_message["Subject"] = subject
        email_message.set_content(body)
        return email_message

    async def send_email(self, subject: str, body: str, to: list[str]):
        """Sends an email asynchronously via aiosmtplib.

        Connects to the SMTP server over TLS and sends the message.

        Args:
            subject: Email subject line.
            body: Plain-text email body.
            to: List of primary recipient addresses.

        Raises:
            RuntimeError: If sending fails.
        """
        email_message: EmailMessage = self._build_message(subject, body, to)

        context = ssl.create_default_context()

        try:
            await aiosmtplib.send(
                email_message,
                hostname=self.host,
                port=self.port,
                start_tls=True,
                username=self.user,
                password=self.password,
                timeout=10,
                tls_context=context,
            )
        except Exception as e:
            raise RuntimeError(f"Async email send failed: {e}")

In [ ]:
async def email_run() -> None:
    """Sends a test email using the AsyncEmailSender."""
    sender = AsyncEmailSender(
        host=SMTP_HOST, port=SMTP_PORT, user=SMTP_USER, password=SMTP_PASS
    )
    await sender.send_email(
        subject="Async Email Test",
        body="Sent via aiosmtplib in production style",
        to=[SMTP_USER],
    )


async def run(selected_client: int) -> None:
    """Dispatches email sending to the selected client.

    Args:
        selected_client: 1 for Raw SMTP, 2 for YaGmail, 3 for aiosmtplib.
    """
    if selected_client == 1:
        sender = SMTPEmailSender(
            host=SMTP_HOST, port=SMTP_PORT, user=SMTP_USER, password=SMTP_PASS
        )
        sender.send_email(
            subject="Test Email",
            body="Hello from production-grade SMTP sender",
            to=[SMTP_USER],
        )
    elif selected_client == 2:
        sender = YaGmailSender(user=SMTP_USER, password=SMTP_PASS)
        sender.send_email(
            to=[SMTP_USER],
            subject="Hello via Yagmail",
            body="<br>".join(
                [
                    "This is a production-style email",
                    "Supports HTML too <b>bold</b>",
                ]
            ),
        )
    elif selected_client == 3:
        await email_run()

In [ ]:
if __name__ == "__main__":
    print("Select one of the clients.")
    print("\n 1. Raw SMTP")
    print("\n 2. YaGmail")
    print("\n 3. Aiosmtplib")
    user_inpput: str = input(">> ").strip()
    if user_inpput.isdigit():
        user_inpput = int(user_inpput)
        if 1 <= int(user_inpput) <= 3:
            await run(selected_client=user_inpput)
        else:
            print("Please select one of the available clients.")
    else:
        print("Enter digit.")